# Evidencia de Codigo - Generacion Dataset Tiles v10/v5b

Notebook consolidado de entrega final para Situacion 2. Raiz evaluable unica: `/workspace/geovision-cali-hf/Entrega_Final/Situacion2`.

## Objetivo

Documentar, de forma limpia y sin ruido historico, el protocolo y codigo base usado para generar el dataset final de 1500 pares imagen-texto de Situacion 2. Este notebook se conserva como evidencia metodologica del codigo empleado; no se re-ejecuta como parte obligatoria de la entrega porque depende de caches Sentinel-2 y artefactos pesados ya auditados.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

ROOT = Path('/workspace/geovision-cali-hf/Entrega_Final/Situacion2')
pd.set_option('display.max_colwidth', 160)


## Dataset Final Auditado

La salida final usada por el modelo v10/v5b esta consolidada en `metricas/dataset_summary.json`. La metadata original completa vive fuera de esta entrega en `outputs/clip_dataset_final_step_by_step/clip_s2_12band_pdf_classes_v5b_high_purity_1500_gsplit/metadata.jsonl`.

In [2]:
with open(ROOT / 'metricas/dataset_summary.json') as f:
    summary = json.load(f)
display(pd.DataFrame([summary]))
display(pd.DataFrame(summary['class_counts'].items(), columns=['class', 'n']))
display(pd.DataFrame(summary['split_counts'].items(), columns=['split', 'n']))


,dataset,source_dataset,n_records,split_strategy,class_counts,split_counts,class_split_counts,scene_counts_by_split,scene_overlap_train_val,scene_overlap_train_test,scene_overlap_val_test,metadata_md5,note
0,clip_s2_12band_pdf_classes_v5b_high_purity_1500_gsplit,/workspace/geovision-cali-hf/outputs/clip_dataset_final_step_by_step/clip_s2_12band_pdf_classes_v5b_high_purity_1500,1500,scene_id_group_holdout_no_scene_overlap,"{'contaminacion_alta_NO2': 300, 'contaminacion_alta_SO2': 300, 'ozono_anomalo': 300, 'suelo_urbano': 300, 'vegetacion_densa': 300}","{'test': 235, 'train': 1000, 'val': 265}","{'contaminacion_alta_NO2': {'test': 44, 'train': 212, 'val': 44}, 'contaminacion_alta_SO2': {'test': 50, 'train': 200, 'val': 50}, 'ozono_anomalo': {'test':...","{'train': 94, 'val': 11, 'test': 10}",0,0,0,be91fc65425e2181b1ddf6d91da878b6,Mismas imagenes y mismo orden que v5b; solo cambia split para evaluar generalizacion por escena/fecha.


,class,n
0,contaminacion_alta_NO2,300
1,contaminacion_alta_SO2,300
2,ozono_anomalo,300
3,suelo_urbano,300
4,vegetacion_densa,300


,split,n
0,test,235
1,train,1000
2,val,265


## Validaciones Criticas

Estas validaciones son las que importan para defensa: 1500 pares, cinco clases balanceadas, split por escena y cero solapamiento entre particiones.

In [3]:
checks = pd.DataFrame([
    {'check': 'n_records == 1500', 'value': summary['n_records'], 'pass': summary['n_records'] == 1500},
    {'check': '5 clases balanceadas con 300 pares', 'value': summary['class_counts'], 'pass': all(v == 300 for v in summary['class_counts'].values())},
    {'check': 'train/val/test = 1000/265/235', 'value': summary['split_counts'], 'pass': summary['split_counts'] == {'train': 1000, 'val': 265, 'test': 235}},
    {'check': 'sin overlap train-val', 'value': summary['scene_overlap_train_val'], 'pass': summary['scene_overlap_train_val'] == 0},
    {'check': 'sin overlap train-test', 'value': summary['scene_overlap_train_test'], 'pass': summary['scene_overlap_train_test'] == 0},
    {'check': 'sin overlap val-test', 'value': summary['scene_overlap_val_test'], 'pass': summary['scene_overlap_val_test'] == 0},
])
display(checks)


,check,value,pass
0,n_records == 1500,1500,True
1,5 clases balanceadas con 300 pares,"{'contaminacion_alta_NO2': 300, 'contaminacion_alta_SO2': 300, 'ozono_anomalo': 300, 'suelo_urbano': 300, 'vegetacion_densa': 300}",True
2,train/val/test = 1000/265/235,"{'test': 235, 'train': 1000, 'val': 265}",True
3,sin overlap train-val,0,True
4,sin overlap train-test,0,True
5,sin overlap val-test,0,True


## Interpretacion de la Auditoria

El dataset final cumple tres condiciones criticas para defender el entrenamiento: tamano suficiente (`1500` pares), balance exacto entre cinco clases (`300` pares por clase) y split por escena sin solapamiento. La particion final `1000/265/235` no es exactamente `70/15/15`; se priorizo holdout por `scene_id` porque reduce mejor el riesgo de fuga espacial/temporal entre train, validacion y test.

## Codigo Base Del Protocolo

El bloque siguiente resume el flujo reproducible empleado. Se deja como codigo documentado, no como ejecucion obligatoria dentro de la entrega final.

In [4]:
# Pseudocodigo fiel al protocolo final v10/v5b.
# Las rutas pesadas se mantienen fuera de Entrega_Final para evitar duplicacion.

from pathlib import Path
import json
import pandas as pd

BASE = Path('/workspace/geovision-cali-hf')
DATASET = BASE / 'outputs/clip_dataset_final_step_by_step/clip_s2_12band_pdf_classes_v5b_high_purity_1500_gsplit'
METADATA = DATASET / 'metadata.jsonl'
SUMMARY = DATASET / 'dataset_summary.json'

def load_metadata(path=METADATA):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

def audit_dataset(summary_path=SUMMARY):
    with open(summary_path, encoding='utf-8') as f:
        s = json.load(f)
    assert s['n_records'] == 1500
    assert all(v == 300 for v in s['class_counts'].values())
    assert s['split_counts'] == {'train': 1000, 'val': 265, 'test': 235}
    assert s['scene_overlap_train_val'] == 0
    assert s['scene_overlap_train_test'] == 0
    assert s['scene_overlap_val_test'] == 0
    return s

# df = load_metadata()
# summary = audit_dataset()


## Decisiones Metodologicas

- Sentinel-5P se usa como pseudo-etiqueta y trazabilidad textual, no como banda ni feature numerica directa del modelo.
- El split se realiza por escena para reducir fuga espacial/temporal entre train, val y test.
- Las clases finales son cinco y quedan balanceadas a 300 pares por clase.
- La evidencia final se valida por `metadata_md5 = be91fc65425e2181b1ddf6d91da878b6`.

La interpretacion correcta es que este notebook prueba el protocolo y la auditoria del dataset final; no intenta reconstruir toda la descarga/procesamiento pesado dentro de la carpeta de entrega.